In [1]:
# Added 11, 12, 13, 14, 15, 16, 17, 18, 19
#  21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31 so far.

In [2]:
import json

file_path = "/Users/ben/Documents/bruegel/data_new/WORKING/INDUSTRY/tuone_v6/src/cached_article_database.jsonl"

# Read and store JSONL data
articles = []
with open(file_path, "r", encoding="utf-8") as file:
    for i, line in enumerate(file):
        data = json.loads(line)
        articles.append(data)

        # Preview only the first 5 lines
        if i < 4:
            print(f"Line {i+1}: {data}")

print(f"\n✅ Loaded {len(articles)} articles in total.")

Line 1: {'title': 'Mitsubishi Chemical expands production capacities', 'paragraphs': {'p1': 'Tokyo-based Mitsubishi Chemical wants to boost up its battery electrolyte business. It plans several strategic steps due to growing demand on the electric vehicle market.', 'p2': 'The company plans to restarting production in a factory based in Britain and doubling its output in the States. Part of the reconstruction plan is also to close a factory in China. The company is said to be even willing to temporarily limit its mobile device battery production.', 'p3': 'Initially, the UK plant has built batteries since 2012. However, the anticipated demand did not arise, so the production lines were stopped in March 2016.asia.nikkei.com', 'p4': 'Your email address will not be published.Required fields are marked*', 'p5': 'Name *', 'p6': 'E-Mail *', 'p7': 'I agree with thePrivacy policy', 'p8': "We have been covering the development of electric mobility with journalistic passion and expertise since 201

In [3]:
import pandas as pd
df = pd.DataFrame(articles)

In [47]:
processed_articles = []
for raw_data in articles:
    # extracting the paragraphs
    paragraphs = [raw_data[key] for key in sorted(raw_data.keys()) if key.startswith("p")]

    #extract the meta-data object
    meta_data = raw_data.get("meta", {})

    # Only process articles where ID starts with "27"
    if meta_data.get("ID", "").startswith("33"):
        article = {
            "title": raw_data.get("title", ""),
            "paragraphs": paragraphs,  # Store paragraphs as a list
            "meta": {
                "ID": meta_data.get("ID", ""),
                "date": meta_data.get("date", ""),
                "url": meta_data.get("url", "")
            }
        }

        processed_articles.append(article)

# Display sample processed articles
display(pd.DataFrame(processed_articles).head())

,title,paragraphs,meta
0,"US Treasury: Rivian, VW, Nissan, Others Lose F...",[{'p1': 'Following its delayed but long-antici...,"{'ID': '3300003', 'date': '18-04-2023', 'url':..."
1,Korean Battery Makers Perplexed by Continuing ...,[{'p1': 'Following a series of Hyundai Motor's...,"{'ID': '3300009', 'date': '16-10-2020', 'url':..."
2,Chinese Companies Leading Global EV Battery Ma...,[{'p1': 'Market research firm SNE Research ann...,"{'ID': '3300010', 'date': '03-08-2022', 'url':..."
3,China's Sichuan Yahua to Supply LiOH to Tesla,[{'p1': 'Chinese lithium producer Sichuan Yahu...,"{'ID': '3300021', 'date': '30-12-2020', 'url':..."
4,Deliveries Begin for Tesla's China-Made Model Y,[{'p1': 'U.S. electric carmaker Tesla Inc. on ...,"{'ID': '3300023', 'date': '19-01-2021', 'url':..."


In [48]:
print(f"Total articles to process: {len(articles)}")
print(f"Total processed articles: {len(processed_articles)}")


Total articles to process: 38310
Total processed articles: 636


In [49]:
#"meta.ID": {"$regex": "^27"}

In [50]:
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
COLLECTION_NAME = os.getenv("MONGO_COLLECTION_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, COLLECTION_NAME]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()

# Create MongoDB client with TLS verification
try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]
    collection = db[COLLECTION_NAME]

    # Verify connection
    client.admin.command('ping')
    print("✅ Connected to MongoDB Atlas!")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}") 
    exit()

✅ Connected to MongoDB Atlas!


In [51]:
import time

total_records = len(processed_articles)
success_count = 0
failures = []

print("\n🚀 Starting data insertion into MongoDB...\n")

for i, article in enumerate(processed_articles, start=1):
    try:
        collection.insert_one(article)
        success_count += 1
        print(f"✅ Processing {i} of {total_records}... Inserted successfully.")
    except Exception as e:
        print(f"❌ Error inserting record {i}: {e}")
        failures.append({"index": i, "error": str(e)})
    time.sleep(0.1)  # Small delay to avoid overwhelming MongoDB (optional)

# Final summary
print(f"\n✅ Successfully inserted {success_count} out of {total_records} records into MongoDB Atlas.")
if failures:
    print(f"⚠️ {len(failures)} records failed to insert. Check error logs.")

# Optional: Save failed records to a log file
if failures:
    with open("mongo_insert_errors.log", "w") as log_file:
        for failure in failures:
            log_file.write(f"Record {failure['index']} - Error: {failure['error']}\n")
    print("⚠️ Failed records have been logged in 'mongo_insert_errors.log'.")


🚀 Starting data insertion into MongoDB...

✅ Processing 1 of 636... Inserted successfully.
✅ Processing 2 of 636... Inserted successfully.
✅ Processing 3 of 636... Inserted successfully.
✅ Processing 4 of 636... Inserted successfully.
✅ Processing 5 of 636... Inserted successfully.
✅ Processing 6 of 636... Inserted successfully.
✅ Processing 7 of 636... Inserted successfully.
✅ Processing 8 of 636... Inserted successfully.
✅ Processing 9 of 636... Inserted successfully.
✅ Processing 10 of 636... Inserted successfully.
✅ Processing 11 of 636... Inserted successfully.
✅ Processing 12 of 636... Inserted successfully.
✅ Processing 13 of 636... Inserted successfully.
✅ Processing 14 of 636... Inserted successfully.
✅ Processing 15 of 636... Inserted successfully.
✅ Processing 16 of 636... Inserted successfully.
✅ Processing 17 of 636... Inserted successfully.
✅ Processing 18 of 636... Inserted successfully.
✅ Processing 19 of 636... Inserted successfully.
✅ Processing 20 of 636... Inserted

In [24]:
#db = client["tuone"]
#collection = db["articles"]
#collection.drop()
#print("🗑️ Collection 'articles' dropped successfully.")

🗑️ Collection 'articles' dropped successfully.
